In [ ]:
# ============================================================
# 04_gold_data_model.ipynb
#
# Purpose:
# Build, persist, reload, inspect, and validate the Gold
# dimensional model using reusable production functions from:
#
# src/gold_data_model.py
# ============================================================


# ============================================================
# CELL 1 — Imports and project configuration
# ============================================================

from pathlib import Path
from pprint import pprint
from IPython.display import display
import importlib
import sys

import pandas as pd


# The notebook may be opened from either:
# 1. the project root, or
# 2. the notebooks directory.
current_directory = Path.cwd().resolve()

if (current_directory / "src").exists():
    PROJECT_ROOT = current_directory

elif (current_directory.parent / "src").exists():
    PROJECT_ROOT = current_directory.parent

else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "The project root must contain the src directory."
    )


# Make the project modules importable by the notebook.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Import and reload the module so notebook reruns use the
# latest saved version of gold_data_model.py.
import src.gold_data_model as gold_data_model

importlib.reload(gold_data_model)

build_gold_data_model = (
    gold_data_model.build_gold_data_model
)

validate_gold_model = (
    gold_data_model.validate_gold_model
)


# Define the project data directories.
DATA_PATH = PROJECT_ROOT / "data"
SILVER_PATH = DATA_PATH / "silver"
GOLD_PATH = DATA_PATH / "gold"


print("Project root:", PROJECT_ROOT)
print("Silver directory:", SILVER_PATH)
print("Gold directory:", GOLD_PATH)

print("\nSilver directory exists:", SILVER_PATH.exists())
print("Gold directory exists:", GOLD_PATH.exists())


# ============================================================
# CELL 2 — Locate the Silver input file
# ============================================================

silver_files = sorted(
    file_path
    for file_path in SILVER_PATH.glob("*.parquet")
    if file_path.is_file()
)

print("Silver Parquet files found:")

for file_path in silver_files:
    print("-", file_path.name)


if not silver_files:
    raise FileNotFoundError(
        "No Silver Parquet file was found in: "
        f"{SILVER_PATH}"
    )


if len(silver_files) > 1:
    raise ValueError(
        "More than one Silver Parquet file was found. "
        "The required transaction file must be selected "
        "explicitly.\n\nFiles found:\n"
        + "\n".join(
            f"- {file_path.name}"
            for file_path in silver_files
        )
    )


silver_input_path = silver_files[0]


print("\nSelected Silver input:")
print(silver_input_path)

print(
    "\nSilver input exists:",
    silver_input_path.exists(),
)


# ============================================================
# CELL 3 — Define the Gold output paths
# ============================================================

merchant_output_path = (
    GOLD_PATH / "dim_merchant.parquet"
)

date_output_path = (
    GOLD_PATH / "dim_date.parquet"
)

transaction_output_path = (
    GOLD_PATH / "fact_transaction.parquet"
)


print("Gold output files:")

print(
    "- Merchant dimension:",
    merchant_output_path,
)

print(
    "- Date dimension:",
    date_output_path,
)

print(
    "- Transaction fact:",
    transaction_output_path,
)


# ============================================================
# CELL 4 — Build and persist the Gold dimensional model
# ============================================================

gold_execution_result = build_gold_data_model(
    silver_input_path=silver_input_path,
    merchant_output_path=merchant_output_path,
    date_output_path=date_output_path,
    transaction_output_path=transaction_output_path,
)


print("Gold data-model execution completed.\n")

pprint(gold_execution_result)


# ============================================================
# CELL 5 — Validate the returned execution metrics
# ============================================================

assert gold_execution_result["status"] == "SUCCESS"

assert (
    gold_execution_result["silver_rows_read"]
    == gold_execution_result[
        "transaction_rows_written"
    ]
)

assert (
    gold_execution_result[
        "merchant_rows_written"
    ]
    > 0
)

assert (
    gold_execution_result[
        "date_rows_written"
    ]
    > 0
)

assert (
    gold_execution_result[
        "transaction_rows_written"
    ]
    > 0
)

assert gold_execution_result[
    "in_memory_validation"
]["is_valid"]

assert gold_execution_result[
    "persisted_validation"
]["is_valid"]


print("Gold execution-metric validation passed.")


# ============================================================
# CELL 6 — Confirm that the output files exist
# ============================================================

expected_gold_files = [
    merchant_output_path,
    date_output_path,
    transaction_output_path,
]


missing_gold_files = [
    file_path
    for file_path in expected_gold_files
    if not file_path.exists()
]


if missing_gold_files:
    raise FileNotFoundError(
        "The following Gold output files were not created:\n"
        + "\n".join(
            str(file_path)
            for file_path in missing_gold_files
        )
    )


print("All expected Gold files were created:")

for file_path in expected_gold_files:
    print(
        "-",
        file_path.name,
        "|",
        file_path.stat().st_size,
        "bytes",
    )


# ============================================================
# CELL 7 — Reload the persisted Gold tables
# ============================================================

dim_merchant_df = pd.read_parquet(
    merchant_output_path
)

dim_date_df = pd.read_parquet(
    date_output_path
)

fact_transaction_df = pd.read_parquet(
    transaction_output_path
)


print(
    "Persisted Merchant rows:",
    len(dim_merchant_df),
)

print(
    "Persisted Date rows:",
    len(dim_date_df),
)

print(
    "Persisted Transaction rows:",
    len(fact_transaction_df),
)


print("\nMerchant columns:")
print(dim_merchant_df.columns.tolist())

print("\nDate columns:")
print(dim_date_df.columns.tolist())

print("\nTransaction columns:")
print(fact_transaction_df.columns.tolist())


# ============================================================
# CELL 8 — Independently validate the persisted model
# ============================================================

silver_df = pd.read_parquet(
    silver_input_path
)

silver_row_count = len(silver_df)


persisted_validation_result = validate_gold_model(
    dim_merchant_df=dim_merchant_df,
    dim_date_df=dim_date_df,
    fact_transaction_df=fact_transaction_df,
    expected_fact_rows=silver_row_count,
)


print("Persisted Gold validation result:\n")

pprint(persisted_validation_result)


assert persisted_validation_result["is_valid"]

assert (
    persisted_validation_result[
        "actual_fact_rows"
    ]
    == silver_row_count
)

assert (
    persisted_validation_result[
        "duplicate_merchant_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "duplicate_date_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "duplicate_transaction_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "missing_merchant_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "missing_date_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "missing_transaction_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "invalid_merchant_foreign_keys"
    ]
    == 0
)

assert (
    persisted_validation_result[
        "invalid_date_foreign_keys"
    ]
    == 0
)


print("Persisted Gold model validation passed.")


# ============================================================
# CELL 9 — Inspect the persisted Gold tables
# ============================================================

print("DIM_MERCHANT")
display(dim_merchant_df.head(10))


print("DIM_DATE")
display(dim_date_df.head(10))


print("FACT_TRANSACTION")
display(fact_transaction_df.head(10))


# ============================================================
# CELL 10 — Final reconciliation summary
# ============================================================

print("=" * 60)
print("GOLD DATA MODEL COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Silver rows read:",
    gold_execution_result["silver_rows_read"],
)

print(
    "Merchant rows written:",
    gold_execution_result[
        "merchant_rows_written"
    ],
)

print(
    "Date rows written:",
    gold_execution_result[
        "date_rows_written"
    ],
)

print(
    "Transaction rows written:",
    gold_execution_result[
        "transaction_rows_written"
    ],
)

print(
    "In-memory validation:",
    gold_execution_result[
        "in_memory_validation"
    ]["is_valid"],
)

print(
    "Persisted validation:",
    gold_execution_result[
        "persisted_validation"
    ]["is_valid"],
)

print("=" * 60)